# Load CARWatch logs and Study Manager results

This notebook demonstrates the package capabilities available after the raw-log and Study Manager loaders. It uses synthetic data from `examples/data` and does not access the ignored `playground` directory.

Register the project environment before opening the notebook with `uv run poe conf_jupyter`, then select the `carwatch` kernel.

In [ ]:
from pathlib import Path

import carwatch as cw

In [ ]:
DATA_DIR = Path("examples/data")
if not DATA_DIR.exists():
    DATA_DIR = Path("data")
DATA_DIR

## Raw app logs

`load_logs` parses semicolon-delimited app logs, converts Unix milliseconds into timezone-aware timestamps, and retains JSON payloads as dictionaries.

In [ ]:
logs = cw.io.load_logs(
    DATA_DIR / "carwatch_demo_VP01_20250515.csv",
    tz="Europe/Berlin",
)
logs[["timestamp", "action", "source_file"]]

## Extract sampling and awakening events

The extraction functions retain the expected sample, the physical tube that was scanned, and any mismatch between them.

In [ ]:
log_samples = cw.logs.extract_samples(logs)
log_awakening = cw.logs.extract_awakening(logs)

log_samples[[
    "sampling_time",
    "sample",
    "sample_scanned",
    "barcode",
    "sample_mismatch",
]]

In [ ]:
log_awakening

## Study Manager export

`load_study_results` retains one row per participant. Its column levels identify the day, sample, and variable without repeating day-level information for every sample.

In [ ]:
study_results = cw.io.load_study_results(
    DATA_DIR / "study_results.csv",
    tz="Europe/Berlin",
)
study_results

## Focused Study Manager views

The helper functions extract day-level awakening information and sample-level observations only when those representations are needed. The original mismatch summary remains available once per day.

In [ ]:
study_awakening = cw.study_manager.extract_awakening(study_results)
study_samples = cw.study_manager.extract_samples(study_results)
study_awakening

In [ ]:
study_samples

## Audit recorded sample swaps

The per-sample mismatch flag is derived from the expected sample and `sample_scanned`.

In [ ]:
mismatches = study_samples.loc[study_samples["sample_mismatch"].fillna(False)]
mismatches[["barcode", "sample_scanned"]]